# Elastic Net Regression — L1 + L2 Regularization

## 1. Definition and intuition
Elastic Net combines L1 and L2 penalties. It can produce sparse coefficients like Lasso while being more stable when predictors are strongly correlated.

## 2. Objective
$$\min_{w,b}\; \frac{1}{2n}\|y-Xw-b\mathbf{1}\|_2^2 + \alpha\,r\|w\|_1 + \frac{\alpha(1-r)}{2}\|w\|_2^2,\quad r\in[0,1].$$

## 3. Why it works: derivation and assumptions
Coordinate descent combines soft-thresholding with an L2-adjusted denominator: $w_j\leftarrow S(x_j^Tr_{-j},n\alpha r)/(x_j^Tx_j+n\alpha(1-r))$. When r=1 the objective becomes Lasso; when r=0 it becomes Ridge-like L2 regularization. The objective is convex; for a positive L2 component it is strictly convex in coefficients under common conditions.

**Assumptions and failure modes:** Use when some sparsity is desired but correlated predictors should be treated more as a group than under pure Lasso. Scaling matters because both penalties act on coefficients.

## 4. Dataset and preprocessing

We use a synthetic regression task with informative features and correlated columns. Predictors are standardized using statistics fit on the training split only. This prevents test-set leakage and makes coefficient penalties comparable across feature scales.

In [1]:
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
rng = np.random.default_rng(SEED)
X, y, true_coef = make_regression(
    n_samples=700, n_features=24, n_informative=8,
    noise=12.0, coef=True, random_state=SEED
)
# Add correlated columns to make the regularization trade-off visible.
X[:, 20] = X[:, 0] * 0.9 + rng.normal(0, 0.15, len(X))
X[:, 21] = X[:, 0] * 0.9 + rng.normal(0, 0.15, len(X))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (525, 24) Test: (175, 24)


## 5. From-scratch implementation

The implementation centers the training data to estimate an unpenalized intercept, then fits coefficients using the objective defined above. It validates shapes and parameter ranges.

In [2]:
def predict_linear(X, intercept, coef):
    return np.asarray(X) @ coef + intercept

def soft_threshold(z, threshold):
    return np.sign(z) * np.maximum(np.abs(z) - threshold, 0.0)

def fit_elastic_net_cd(X, y, alpha=0.08, l1_ratio=0.5, max_iter=20000, tol=1e-8):
    X = np.asarray(X, dtype=float); y = np.asarray(y, dtype=float)
    if X.ndim != 2 or y.ndim != 1 or len(X) != len(y) or len(X) == 0:
        raise ValueError("X must be non-empty 2D and y must be matching 1D")
    if alpha < 0 or not 0 <= l1_ratio <= 1 or max_iter < 1 or tol < 0:
        raise ValueError("Invalid hyperparameters")
    xm, ym = X.mean(axis=0), y.mean()
    Xc, yc = X - xm, y - ym
    n, d = Xc.shape
    w = np.zeros(d)
    norms = np.sum(Xc * Xc, axis=0)
    l1 = n * alpha * l1_ratio
    l2 = n * alpha * (1 - l1_ratio)
    for iteration in range(max_iter):
        max_change = 0.0
        for j in range(d):
            if norms[j] + l2 == 0:
                w[j] = 0.0
                continue
            residual_without_j = yc - Xc @ w + Xc[:, j] * w[j]
            new_wj = soft_threshold(Xc[:, j] @ residual_without_j, l1) / (norms[j] + l2)
            max_change = max(max_change, abs(new_wj - w[j]))
            w[j] = new_wj
        if max_change < tol:
            break
    return ym - xm @ w, w, iteration + 1

alpha, l1_ratio = 0.08, 0.5
b_scratch, w_scratch, n_iter = fit_elastic_net_cd(
    X_train, y_train, alpha=alpha, l1_ratio=l1_ratio
)
pred_scratch = predict_linear(X_test, b_scratch, w_scratch)
print("Iterations:", n_iter, "nonzero coefficients:", int(np.count_nonzero(np.abs(w_scratch) > 1e-8)))
print("From-scratch MSE:", mean_squared_error(y_test, pred_scratch))


Iterations: 125 nonzero coefficients: 22
From-scratch MSE: 187.1892007852164


## 6. Compare with scikit-learn

Compare predictions, not just coefficient vectors. Coefficients can be affected by solver tolerance and collinearity. The library and scratch implementations below use matching objective conventions.

In [3]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model = ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=50000, tol=1e-8, selection="cyclic").fit(X_train, y_train)
pred_sklearn = model.predict(X_test)
print("scikit-learn MSE:", mean_squared_error(y_test, pred_sklearn))
print("scikit-learn nonzero coefficients:", int(np.count_nonzero(np.abs(model.coef_) > 1e-8)))
print("Max prediction difference:", np.max(np.abs(pred_scratch - pred_sklearn)))
assert np.allclose(pred_scratch, pred_sklearn, atol=5e-3, rtol=5e-3)


scikit-learn MSE: 187.18919915417055
scikit-learn nonzero coefficients: 22
Max prediction difference: 3.982048269790539e-06


## 7. Hyperparameters

`alpha` controls overall penalty strength; `l1_ratio` controls the mix. Values near 1 emphasize sparsity; values near 0 emphasize L2 shrinkage. Tune both using cross-validation.

## 8. Best practices

Scale inputs; compare several `l1_ratio` values; use CV; monitor convergence; inspect how stable coefficients are across folds; consider Ridge if prediction matters more than sparsity.

## 9. Time/space complexity

Cyclic coordinate descent is approximately O(qnd) for q full coordinate sweeps on dense inputs, with convergence depending on feature correlations and tolerances. Inference O(md), model storage O(d).

## 10. Scaling to millions of samples

Use float32 when accuracy allows, sparse-aware solvers for sparse matrices, and iterative/minibatch methods for large n. Avoid materializing copies unnecessarily. For very high dimensions, choose solvers based on the relationship between n and d. Benchmark quality, runtime and peak memory together.

## 11. GPU suitability

A GPU is most promising when the implementation is dominated by large dense matrix operations. Coordinate descent updates one coefficient at a time, so it may be less GPU-friendly than proximal-gradient optimization. Always include data transfer and device memory in timing.

In [4]:
# GPU implementation options include proximal gradient, which parallelizes matrix multiplications.
# This notebook leaves GPU benchmarking conditional because the best choice depends on data size and device memory.
try:
    import torch
    if torch.cuda.is_available():
        print("CUDA is available. For a meaningful benchmark, compare proximal-gradient Elastic Net against optimized CPU coordinate descent on a larger dense dataset.")
    else:
        print("CUDA unavailable; keep the CPU coordinate-descent implementation as the baseline.")
except ImportError:
    print("PyTorch is optional and is not installed.")


CUDA unavailable; keep the CPU coordinate-descent implementation as the baseline.


## 12. Correctness tests and synthesis

The regularizer trades coefficient flexibility for stability. Ridge shrinks coefficients continuously, Lasso can make them exactly zero, and Elastic Net interpolates between these behaviors. The validation strategy matters as much as the optimizer.

In [5]:
assert np.isfinite(pred_scratch).all()
assert len(pred_scratch) == len(y_test)
assert mean_squared_error(y_test, pred_scratch) < np.var(y_test)
try:
    fit_elastic_net_cd(np.array([1, 2, 3]), np.array([1, 2, 3]))
except ValueError:
    pass
else:
    raise AssertionError("Expected invalid X dimensionality to raise")
print("All Elastic Net checks passed.")


All Elastic Net checks passed.
